<a href="https://colab.research.google.com/github/simoesya/ProjetoFinal/blob/main/Projeto_RAG_TriviaQA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/Projeto_RAG_TriviaQA")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

for pasta in ["data", "results", "checkpoints"]:
    (PROJECT_DIR / pasta).mkdir(exist_ok=True)

print("Pasta do projeto criada:", PROJECT_DIR)

Pasta do projeto criada: /content/drive/MyDrive/Projeto_RAG_TriviaQA


In [ ]:
import urllib.request

url = "https://nlp.cs.washington.edu/triviaqa/data/triviaqa-rc.tar.gz"
arquivo = PROJECT_DIR / "data" / "triviaqa-rc.tar.gz"

if arquivo.exists():
    print("O ficheiro já existe:", arquivo)
else:
    print("A descarregar o TriviaQA. Aguarda...")
    temporario = arquivo.with_suffix(".part")
    urllib.request.urlretrieve(url, temporario)
    temporario.rename(arquivo)
    print("Download concluído!")

print(f"Tamanho: {arquivo.stat().st_size / 1024**3:.2f} GB")

A descarregar o TriviaQA. Aguarda...
Download concluído!
Tamanho: 2.48 GB


In [ ]:
import tarfile
from pathlib import PurePosixPath

DATA_DIR = PROJECT_DIR / "data" / "triviaqa"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("A extrair os ficheiros da Wikipédia. Aguarda...")

with tarfile.open(arquivo, "r:gz") as tar:
    for membro in tar:
        partes = PurePosixPath(membro.name).parts

        perguntas = membro.name.endswith((
            "qa/wikipedia-train.json",
            "qa/verified-wikipedia-dev.json"
        ))
        documento = "evidence" in partes and "wikipedia" in partes

        if membro.isfile() and (perguntas or documento):
            tar.extract(membro, path=DATA_DIR, filter="data")

print("Extração concluída!")

for ficheiro in DATA_DIR.rglob("*wikipedia*.json"):
    print(ficheiro.relative_to(DATA_DIR))

A extrair os ficheiros da Wikipédia. Aguarda...
Extração concluída!
qa/verified-wikipedia-dev.json
qa/wikipedia-train.json


In [2]:
import json
from pathlib import Path

DATA_DIR = Path(
    "/content/drive/MyDrive/Projeto_RAG_TriviaQA/data/triviaqa"
)

with open(DATA_DIR / "qa/wikipedia-train.json", encoding="utf-8") as f:
    perguntas = json.load(f)["Data"]

with open(DATA_DIR / "qa/verified-wikipedia-dev.json", encoding="utf-8") as f:
    teste = json.load(f)["Data"]

# As primeiras 7900 perguntas são para validação.
validacao = perguntas[:7900]
treino = perguntas[7900:]

assert len(validacao) == 7900

print("Perguntas de treino:", len(treino))
print("Perguntas de validação:", len(validacao))
print("Perguntas de teste:", len(teste))

exemplo = validacao[0]
print("\nPergunta:", exemplo["Question"])
print("Resposta correta:", exemplo["Answer"]["Value"])
print("Documentos associados:", len(exemplo.get("EntityPages", [])))

Perguntas de treino: 53988
Perguntas de validação: 7900
Perguntas de teste: 318

Pergunta: Where in England was Dame Judi Dench born?
Resposta correta: York
Documentos associados: 2


In [3]:
exemplo = validacao[0]
documento = exemplo["EntityPages"][0]

caminho = DATA_DIR / "evidence" / "wikipedia" / documento["Filename"]

print("Pergunta:", exemplo["Question"])
print("Resposta correta:", exemplo["Answer"]["Value"])
print("Ficheiro:", caminho)

with open(caminho, encoding="utf-8") as f:
    texto = f.read()

print("\nNúmero de palavras:", len(texto.split()))
print("\nInício do documento:\n")
print(texto[:2000])


Pergunta: Where in England was Dame Judi Dench born?
Resposta correta: York
Ficheiro: /content/drive/MyDrive/Projeto_RAG_TriviaQA/data/triviaqa/evidence/wikipedia/England.txt

Número de palavras: 13835

Início do documento:

England is a country that is part of the United Kingdom.   It shares land borders with Scotland to the north and Wales to the west. The Irish Sea lies northwest of England and the Celtic Sea lies to the southwest. England is separated from continental Europe by the North Sea to the east and the English Channel to the south. The country covers much of the central and southern part of the island of Great Britain, which lies in the North Atlantic; and includes over 100 smaller islands such as the Isles of Scilly, and the Isle of Wight.

The area now called England was first inhabited by modern humans during the Upper Palaeolithic period, but takes its name from the Angles, one of the Germanic tribes who settled during the 5th and 6th centuries. England became a unifie

In [4]:
def dividir_em_excertos(texto, tamanho=200, sobreposicao=50):
    if not 0 <= sobreposicao < tamanho:
        raise ValueError("A sobreposição deve estar entre 0 e tamanho - 1.")

    palavras = texto.split()
    excertos = []
    passo = tamanho - sobreposicao

    for inicio in range(0, len(palavras), passo):
        excertos.append(" ".join(palavras[inicio:inicio + tamanho]))

        if inicio + tamanho >= len(palavras):
            break

    return excertos

excertos = dividir_em_excertos(texto)

print("Total de excertos:", len(excertos))
print("Palavras no primeiro excerto:", len(excertos[0].split()))
print("\nPrimeiro excerto:\n")
print(excertos[0])

Total de excertos: 92
Palavras no primeiro excerto: 200

Primeiro excerto:

England is a country that is part of the United Kingdom. It shares land borders with Scotland to the north and Wales to the west. The Irish Sea lies northwest of England and the Celtic Sea lies to the southwest. England is separated from continental Europe by the North Sea to the east and the English Channel to the south. The country covers much of the central and southern part of the island of Great Britain, which lies in the North Atlantic; and includes over 100 smaller islands such as the Isles of Scilly, and the Isle of Wight. The area now called England was first inhabited by modern humans during the Upper Palaeolithic period, but takes its name from the Angles, one of the Germanic tribes who settled during the 5th and 6th centuries. England became a unified state in the 10th century, and since the Age of Discovery, which began during the 15th century, has had a significant cultural and legal impact on the

In [5]:
excertos_exemplo = []

for documento in validacao[0]["EntityPages"]:
    caminho = DATA_DIR / "evidence" / "wikipedia" / documento["Filename"]

    with open(caminho, encoding="utf-8") as f:
        conteudo = f.read()

    partes = dividir_em_excertos(conteudo)

    for numero, parte in enumerate(partes):
        excertos_exemplo.append({
            "ficheiro": documento["Filename"],
            "numero": numero,
            "texto": parte
        })

    print(f"{documento['Filename']}: {len(partes)} excertos")

print("\nTotal de excertos para esta pergunta:", len(excertos_exemplo))

England.txt: 92 excertos
Judi_Dench.txt: 39 excertos

Total de excertos para esta pergunta: 131


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pergunta = validacao[0]["Question"]
textos = [excerto["texto"] for excerto in excertos_exemplo]

vetorizador = TfidfVectorizer(stop_words="english")
matriz = vetorizador.fit_transform(textos)
vetor_pergunta = vetorizador.transform([pergunta])

pontuacoes = cosine_similarity(vetor_pergunta, matriz).ravel()
melhores = pontuacoes.argsort()[::-1][:5]

print("Pergunta:", pergunta)

for posicao, indice in enumerate(melhores, start=1):
    excerto = excertos_exemplo[indice]

    print(f"\n--- Resultado {posicao} ---")
    print("Ficheiro:", excerto["ficheiro"])
    print("Pontuação:", round(float(pontuacoes[indice]), 3))
    print(excerto["texto"])

Pergunta: Where in England was Dame Judi Dench born?

--- Resultado 1 ---
Ficheiro: Judi_Dench.txt
Pontuação: 0.246
Screen Actors Guild Awards, two Golden Globes, an Academy Award, and a Tony Award. She has also received the BAFTA Fellowship (2001) and the Special Olivier Award (2004). In June 2011, she received a fellowship from the British Film Institute (BFI). Dench is also a Fellow of the Royal Society of Arts (FRSA). Early life Dench was born in Heworth, North Riding of Yorkshire. Her mother, Eleanora Olive (née Jones), was born in Dublin. Her father, Reginald Arthur Dench, a doctor, was born in Dorset, and later moved to Dublin, where he was raised. He met Dench's mother while he was studying medicine at Trinity College, Dublin. Dench attended The Mount School, a Quaker independent secondary school in York, and became a Quaker. Her brothers, one of whom was actor Jeffery Dench, were born in Tyldesley, Lancashire. Her niece, Emma Dench, is a Roman historian and professor previousl

In [11]:
from pathlib import Path
import subprocess
import sys
import importlib

PROJECT_DIR = Path("/content/drive/MyDrive/Projeto_RAG_TriviaQA")
REPO_DIR = Path("/content/triviaqa_official")

# Descarregar o repositório completo, incluindo os módulos auxiliares.
if not (REPO_DIR / ".git").exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/mandarjoshi90/triviaqa.git",
            str(REPO_DIR)
        ],
        check=True
    )

# Permitir que o Python encontre os módulos do repositório.
sys.path.insert(0, str(REPO_DIR))
importlib.invalidate_caches()

from evaluation import triviaqa_evaluation as scorer

print("Avaliador oficial carregado.")
print("Exemplo de normalização:", scorer.normalize_answer("The York!"))
print("Aliases disponíveis:", validacao[0]["Answer"]["NormalizedAliases"])

Avaliador oficial carregado.
Exemplo de normalização: york
Aliases disponíveis: ['york yorkshire', 'eoferwic', 'park grove primary school', 'park grove school', 'weather in york', 'park grove 1895', 'eoforwic', 'county borough of york', 'york uk', 'un locode gbyrk', 'city of york', 'york england', 'york ua', 'york ham', 'york', 'yorkish', 'yoisk', 'york north yorkshire']


In [12]:
aliases = {
    scorer.normalize_answer(alias)
    for alias in validacao[0]["Answer"]["NormalizedAliases"]
}
aliases.discard("")

def contem_resposta(texto, aliases):
    texto_normalizado = " " + scorer.normalize_answer(texto) + " "
    return any(
        " " + alias + " " in texto_normalizado
        for alias in aliases
    )

# Ordenar todos os excertos pela pontuação da pesquisa TF-IDF.
ranking = pontuacoes.argsort()[::-1]

for k in [1, 5, 10]:
    encontrou = any(
        contem_resposta(excertos_exemplo[int(i)]["texto"], aliases)
        for i in ranking[:k]
    )

    print(f"Acerto de recuperação @{k}: {'SIM' if encontrou else 'NÃO'}")

Acerto de recuperação @1: SIM
Acerto de recuperação @5: SIM
Acerto de recuperação @10: SIM


In [13]:
import json
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/Projeto_RAG_TriviaQA")
pasta_divisoes = PROJECT_DIR / "data" / "splits"
pasta_divisoes.mkdir(parents=True, exist_ok=True)

conjuntos = {
    "treino": treino,
    "validacao": validacao,
    "teste": teste
}

# Confirmar que nenhuma pergunta aparece em conjuntos diferentes.
ids = {
    nome: {p["QuestionId"] for p in perguntas}
    for nome, perguntas in conjuntos.items()
}

assert ids["treino"].isdisjoint(ids["validacao"])
assert ids["treino"].isdisjoint(ids["teste"])
assert ids["validacao"].isdisjoint(ids["teste"])

for nome, perguntas in conjuntos.items():
    destino = pasta_divisoes / f"{nome}.json"

    with open(destino, "w", encoding="utf-8") as f:
        json.dump(
            perguntas, f,
            ensure_ascii=False
        )

    print(f"{nome}: {len(perguntas)} perguntas guardadas")

print("\nDivisões guardadas e sem perguntas em comum.")

treino: 53988 perguntas guardadas
validacao: 7900 perguntas guardadas
teste: 318 perguntas guardadas

Divisões guardadas e sem perguntas em comum.


In [14]:
def pesquisar_excertos(pergunta, k=10):
    candidatos = []

    # Evitar ler documentos repetidos na mesma pergunta.
    ficheiros = dict.fromkeys(
        doc["Filename"] for doc in pergunta["EntityPages"]
    )

    for ficheiro in ficheiros:
        caminho = DATA_DIR / "evidence" / "wikipedia" / ficheiro

        with open(caminho, encoding="utf-8") as f:
            texto = f.read()

        for numero, excerto in enumerate(dividir_em_excertos(texto)):
            candidatos.append({
                "ficheiro": ficheiro,
                "numero": numero,
                "texto": excerto
            })

    if not candidatos:
        return []

    vetorizador = TfidfVectorizer(stop_words="english")
    matriz = vetorizador.fit_transform(
        [item["texto"] for item in candidatos]
    )
    consulta = vetorizador.transform([pergunta["Question"]])

    scores = cosine_similarity(consulta, matriz).ravel()
    indices = scores.argsort()[::-1][:k]

    return [
        {**candidatos[int(i)], "score": float(scores[i])}
        for i in indices
    ]

# Testar noutra pergunta de validação.
resultados = pesquisar_excertos(validacao[1])

print("Pergunta:", validacao[1]["Question"])
print("Excertos recuperados:", len(resultados))

if resultados:
    print("\nPrimeiro resultado:", resultados[0]["ficheiro"])
    print(resultados[0]["texto"])

Pergunta: From which country did Angola achieve independence in 1975?
Excertos recuperados: 10

Primeiro resultado: Angola.txt
lasted several decades (with some interludes). It claimed millions of lives and produced many refugees; it didn't end until 2002. Following negotiations held in Portugal, itself experiencing severe social and political turmoil and uncertainty due to the April 1974 revolution, Angola's three main guerrilla groups agreed to establish a transitional government in January 1975. Within two months, however, the FNLA, MPLA and UNITA had started fighting each other and the country began splitting into zones controlled by rival armed political groups. The MPLA gained control of the capital Luanda and much of the rest of the country. With the support of the United States, Zaïre and South Africa intervened militarily in favour of the FNLA and UNITA with the intention of taking Luanda before the declaration of independence. In response, Cuba intervened in favor of the MPLA

In [15]:
from tqdm.auto import tqdm
import json

avaliacao_inicial = []

for pergunta in tqdm(validacao[:20], desc="A avaliar"):
    recuperados = pesquisar_excertos(pergunta, k=10)

    aliases = {
        scorer.normalize_answer(alias)
        for alias in pergunta["Answer"]["NormalizedAliases"]
    }
    aliases.add(scorer.normalize_answer(pergunta["Answer"]["Value"]))
    aliases.discard("")

    registo = {
        "id": pergunta["QuestionId"],
        "pergunta": pergunta["Question"],
        "resultados": recuperados
    }

    for k in [1, 5, 10]:
        registo[f"acerto@{k}"] = any(
            contem_resposta(item["texto"], aliases)
            for item in recuperados[:k]
        )

    avaliacao_inicial.append(registo)

for k in [1, 5, 10]:
    acertos = sum(r[f"acerto@{k}"] for r in avaliacao_inicial)
    total = len(avaliacao_inicial)
    print(f"Recall@{k}: {acertos / total:.1%} ({acertos}/{total})")

destino = PROJECT_DIR / "results"
destino.mkdir(parents=True, exist_ok=True)

with open(destino / "tfidf_validacao_20.json", "w", encoding="utf-8") as f:
    json.dump(avaliacao_inicial, f, ensure_ascii=False, indent=2)

print("\nResultados guardados.")

A avaliar:   0%|          | 0/20 [00:00<?, ?it/s]

Recall@1: 60.0% (12/20)
Recall@5: 80.0% (16/20)
Recall@10: 90.0% (18/20)

Resultados guardados.


In [16]:
perguntas_por_id = {
    p["QuestionId"]: p for p in validacao[:20]
}

for resultado in avaliacao_inicial:
    if resultado["acerto@10"]:
        continue

    pergunta = perguntas_por_id[resultado["id"]]

    print("=" * 70)
    print("Pergunta:", pergunta["Question"])
    print("Resposta correta:", pergunta["Answer"]["Value"])
    print("Aliases:", pergunta["Answer"]["NormalizedAliases"])

    # Verificar se a resposta existe em algum excerto dos documentos.
    aliases = {
        scorer.normalize_answer(a)
        for a in pergunta["Answer"]["NormalizedAliases"]
    }
    aliases.add(scorer.normalize_answer(pergunta["Answer"]["Value"]))
    aliases.discard("")

    todos = pesquisar_excertos(pergunta, k=100000)
    posicao = next(
        (
            i for i, item in enumerate(todos, start=1)
            if contem_resposta(item["texto"], aliases)
        ),
        None
    )

    print("Primeira posição com a resposta:", posicao)

    if resultado["resultados"]:
        print("\nPrimeiro excerto recuperado:")
        print(resultado["resultados"][0]["texto"])

    print()

Pergunta: Which was the first European country to abolish capital punishment?
Resposta correta: Norway
Aliases: ['norwegen', 'kongeriket norge', 'norway', 'republic of norway', 'noorwegen', 'norvege', 'mainland norway', 'kingdom of norway', 'sport in norway', 'noreg', 'noruega', 'norwegia', 'noregur', 'royal kingdom of norway', 'name of norway', 'kongeriket noreg', 'norwegian kingdom', 'etymology of norway', 'norvège', 'iso 3166 1 no', 'norwegian state']
Primeira posição com a resposta: 23

Primeiro excerto recuperado:
to life is the most important, and capital punishment violates it without necessity and inflicts to the condemned a psychological torture. Human rights activists oppose the death penalty, calling it "cruel, inhuman, and degrading punishment". Amnesty International considers it to be "the ultimate, irreversible denial of Human Rights". See also: Capital punishment abolition chronology. Contemporary use Capital punishment by country Most countries including almost all Firs

In [17]:
diagnostico = []

for resultado in avaliacao_inicial:
    if resultado["acerto@10"]:
        continue

    pergunta = perguntas_por_id[resultado["id"]]

    aliases = {
        scorer.normalize_answer(a)
        for a in pergunta["Answer"]["NormalizedAliases"]
    }
    aliases.add(scorer.normalize_answer(pergunta["Answer"]["Value"]))
    aliases.discard("")

    todos = pesquisar_excertos(pergunta, k=100000)

    correspondencia = next(
        (
            (posicao, item)
            for posicao, item in enumerate(todos, start=1)
            if contem_resposta(item["texto"], aliases)
        ),
        None
    )

    diagnostico.append({
        "id": pergunta["QuestionId"],
        "pergunta": pergunta["Question"],
        "resposta": pergunta["Answer"]["Value"],
        "primeira_posicao_com_alias": (
            correspondencia[0] if correspondencia else None
        ),
        "excerto_com_alias": (
            correspondencia[1] if correspondencia else None
        ),
        "primeiro_resultado": (
            resultado["resultados"][0]
            if resultado["resultados"] else None
        )
    })

destino = PROJECT_DIR / "results" / "diagnostico_tfidf_20.json"

with open(destino, "w", encoding="utf-8") as f:
    json.dump(diagnostico, f, ensure_ascii=False, indent=2)

print("Diagnóstico guardado.")

for caso in diagnostico:
    if caso["excerto_com_alias"]:
        print("\nPergunta:", caso["pergunta"])
        print("Excerto com alias:")
        print(caso["excerto_com_alias"]["texto"])

Diagnóstico guardado.

Pergunta: Which was the first European country to abolish capital punishment?
Excerto com alias:
others actively use it today. The death penalty there remains a contentious issue which is hotly debated. In abolitionist countries, the debate is sometimes revived by particularly brutal murders though few countries have brought it back after abolishing it. However, a spike in serious, violent crimes, such as murders or terrorist attacks, has prompted some countries (such as Sri Lanka and Jamaica) to effectively end the moratorium on the death penalty. In retention countries, the debate is sometimes revived when a miscarriage of justice has occurred though this tends to cause legislative efforts to improve the judicial process rather than to abolish the death penalty. Modern-day public opinion The public opinion on the death penalty varies considerably by country and by the crime in question. Countries where a majority of people are against execution include New Zeal

In [18]:
avaliacao_200 = []

for pergunta in tqdm(validacao[:200], desc="A avaliar 200 perguntas"):
    recuperados = pesquisar_excertos(pergunta, k=10)

    aliases = {
        scorer.normalize_answer(a)
        for a in pergunta["Answer"]["NormalizedAliases"]
    }
    aliases.add(scorer.normalize_answer(pergunta["Answer"]["Value"]))
    aliases.discard("")

    registo = {
        "id": pergunta["QuestionId"],
        "pergunta": pergunta["Question"],
        "resultados": recuperados
    }

    for k in [1, 5, 10]:
        registo[f"acerto@{k}"] = any(
            contem_resposta(item["texto"], aliases)
            for item in recuperados[:k]
        )

    avaliacao_200.append(registo)

    # Guardar o progresso a cada 20 perguntas.
    if len(avaliacao_200) % 20 == 0:
        destino = PROJECT_DIR / "results" / "tfidf_validacao_200.json"
        with open(destino, "w", encoding="utf-8") as f:
            json.dump(avaliacao_200, f, ensure_ascii=False, indent=2)

for k in [1, 5, 10]:
    acertos = sum(r[f"acerto@{k}"] for r in avaliacao_200)
    print(f"Recall@{k}: {acertos / len(avaliacao_200):.1%}")

print("\nResultados guardados.")

A avaliar 200 perguntas:   0%|          | 0/200 [00:00<?, ?it/s]

Recall@1: 58.0%
Recall@5: 83.0%
Recall@10: 87.0%

Resultados guardados.


In [19]:
%pip -q install rank-bm25

In [20]:
import re
from rank_bm25 import BM25Okapi

def tokenizar_bm25(texto):
    return re.findall(r"\w+", texto.lower())

def pesquisar_bm25(pergunta, k=10):
    candidatos = []

    ficheiros = dict.fromkeys(
        doc["Filename"] for doc in pergunta["EntityPages"]
    )

    for ficheiro in ficheiros:
        caminho = DATA_DIR / "evidence" / "wikipedia" / ficheiro

        with open(caminho, encoding="utf-8") as f:
            texto = f.read()

        for numero, excerto in enumerate(dividir_em_excertos(texto)):
            if tokenizar_bm25(excerto):
                candidatos.append({
                    "ficheiro": ficheiro,
                    "numero": numero,
                    "texto": excerto
                })

    if not candidatos:
        return []

    indice = BM25Okapi([
        tokenizar_bm25(item["texto"])
        for item in candidatos
    ])

    scores = indice.get_scores(tokenizar_bm25(pergunta["Question"]))
    melhores = scores.argsort()[::-1][:k]

    return [
        {**candidatos[int(i)], "score": float(scores[i])}
        for i in melhores
    ]

avaliacao_bm25_200 = []

for pergunta in tqdm(validacao[:200], desc="A avaliar BM25"):
    recuperados = pesquisar_bm25(pergunta)

    aliases = {
        scorer.normalize_answer(a)
        for a in pergunta["Answer"]["NormalizedAliases"]
    }
    aliases.add(scorer.normalize_answer(pergunta["Answer"]["Value"]))
    aliases.discard("")

    registo = {
        "id": pergunta["QuestionId"],
        "pergunta": pergunta["Question"],
        "resultados": recuperados
    }

    for k in [1, 5, 10]:
        registo[f"acerto@{k}"] = any(
            contem_resposta(item["texto"], aliases)
            for item in recuperados[:k]
        )

    avaliacao_bm25_200.append(registo)

    if len(avaliacao_bm25_200) % 20 == 0:
        destino = PROJECT_DIR / "results" / "bm25_validacao_200.json"
        with open(destino, "w", encoding="utf-8") as f:
            json.dump(
                avaliacao_bm25_200, f,
                ensure_ascii=False, indent=2
            )

print("\nComparação nas mesmas 200 perguntas:")
for k in [1, 5, 10]:
    tfidf = sum(r[f"acerto@{k}"] for r in avaliacao_200) / len(avaliacao_200)
    bm25 = sum(r[f"acerto@{k}"] for r in avaliacao_bm25_200) / len(avaliacao_bm25_200)

    print(f"Recall@{k}: TF-IDF = {tfidf:.1%} | BM25 = {bm25:.1%}")

print("\nResultados guardados.")

A avaliar BM25:   0%|          | 0/200 [00:00<?, ?it/s]


Comparação nas mesmas 200 perguntas:
Recall@1: TF-IDF = 58.0% | BM25 = 53.5%
Recall@5: TF-IDF = 83.0% | BM25 = 82.5%
Recall@10: TF-IDF = 87.0% | BM25 = 88.5%

Resultados guardados.


In [21]:
import importlib.metadata
import subprocess
import json

configuracao = {
    "dataset": "TriviaQA RC — Wikipedia",
    "validacao": "Primeiras 7900 perguntas de wikipedia-train.json",
    "amostra_experimental": "Primeiras 200 perguntas da validação",
    "ambito_pesquisa": "Documentos associados a cada pergunta",
    "tamanho_excerto_palavras": 200,
    "sobreposicao_palavras": 50,
    "tfidf": {
        "stop_words": "english",
        "restantes_parametros": "Defaults do scikit-learn"
    },
    "bm25": {
        "tokenizacao": r"\w+ em minúsculas",
        "k1": 1.5,
        "b": 0.75,
        "epsilon": 0.25
    },
    "metrica": "Presença de alias normalizado com limites de palavras",
    "versoes": {
        pacote: importlib.metadata.version(pacote)
        for pacote in ["scikit-learn", "rank-bm25", "numpy"]
    },
    "commit_avaliador_oficial": subprocess.check_output(
        ["git", "-C", "/content/triviaqa_official", "rev-parse", "HEAD"],
        text=True
    ).strip()
}

destino = PROJECT_DIR / "results" / "configuracao_pesquisa_200.json"

with open(destino, "w", encoding="utf-8") as f:
    json.dump(configuracao, f, ensure_ascii=False, indent=2)

print("Configuração guardada.")

Configuração guardada.
